# Messages between ranks

Imagine a group of workers sharing a computation. Each has its own part to do, but
making progress may require a value from a neighbour, a new job, or a result another
worker has produced. Those messages connect the workers' separate actions into one program.

In this lesson, you'll build that coordination using **point-to-point communication**:
individual sends and receives between workers. You'll work out who needs to talk to
whom, how to receive the right message, and how every worker knows when it can finish.
Sometimes the sender is known in advance; sometimes you'll want whichever worker is
ready, or a result with a particular tag.

These decisions matter even when the local computation is tiny. A worker can calculate
the right answer and still leave the program stuck because another worker is waiting
for a message that will never come. We'll use small computations so you can focus on
how the workers make progress together.


In [ ]:
import flock

from dist_puzzles import check
from dist_puzzles.lessons.p2p import (
    check_collect_reports,
    check_receive_selectively,
    check_ring_shift,
    check_task_queue,
)


## 1. Pass it on

Imagine workers arranged in a ring, each holding a value that its next neighbour needs.
Every rank sends its own value forward and receives the value from the rank behind it.
The last rank sends back to rank 0, closing the ring.

```text
Values before the exchange:
  Rank 0: 10    Rank 1: 20    Rank 2: 30    Rank 3: 40

Messages:
  Rank 0 -- 10 --> Rank 1 -- 20 --> Rank 2 -- 30 --> Rank 3
     ^                                               |
     +-------------------- 40 -----------------------+

Values returned, in rank order: [40, 10, 20, 30]
```

Implement `ring_shift(x)`. Send your integer to the next rank and return the integer
received from the previous rank. Use only point-to-point messages along the ring and
finish every exchange. Your solution should work with any number of ranks greater than one.

<details>
<summary>Hint: getting started</summary>

If everyone starts with a blocking send, nobody reaches a receive.

</details>


In [ ]:
async def ring_shift(x: int) -> int:
    raise NotImplementedError

In [ ]:
inputs = [10, 20, 30, 40]
expected = [40, 10, 20, 30]

run = flock.distribute(workers=4, seed=0, trace=True)(ring_shift)
results, trace = run(flock.per_rank(inputs))

In [ ]:
check(results, expected)
results

In [ ]:
check_ring_shift(ring_shift)

## 2. Collect reports

Suppose each worker has measured a value and rank 0 needs to collect their reports.
Workers can report in any order, but rank 0 must keep track of who reported each value.

```text
Reports to collect:
  Worker 1: 12    Worker 2: -3    Worker 3: 12

One possible arrival order:
  Worker 2 -- -3 --> Rank 0
  Worker 1 -- 12 --> Rank 0
  Worker 3 -- 12 --> Rank 0

Rank 0 returns: {1: 12, 2: -3, 3: 12}
```

Implement `collect_reports(report)`. Return `{worker_rank: report}` on rank 0 and
`None` on workers.
Use point-to-point communication from workers to rank 0 and finish every exchange.
There are at least two ranks.

<details>
<summary>Hint: accepting reports</summary>

`await flock.recv(flock.ANY_SOURCE)` accepts a message from any worker. It returns only
the payload, so include the sender's rank if you need to identify it.

</details>


In [ ]:
async def collect_reports(report: int | None) -> dict[int, int] | None:
    if flock.get_rank() == 0:
        assert report is None
        raise NotImplementedError

    assert report is not None
    raise NotImplementedError


In [ ]:
inputs = [None, 12, -3, 12]
run = flock.distribute(workers=4, seed=0, trace=True)(collect_reports)
results, trace = run(flock.per_rank(inputs))
check(results, [{1: 12, 2: -3, 3: 12}, None, None, None])
results


In [ ]:
check_collect_reports(collect_reports)

## 3. Who's done?

Imagine a list of jobs waiting to be processed by several workers. Some workers may
finish sooner than others, so dividing the list in advance could leave workers idle
while others still have jobs to do. Rank 0 will hold the list and assign another job
whenever a worker becomes available.

Here the jobs are `[2, 3, 5, 6]`, and computing a job means squaring its value.
One possible exchange looks like this:

```text
Rank 0 assigns:    job 2 --> Worker 1
                   job 3 --> Worker 2

Worker 2 replies:  9     --> Rank 0    (result for job 3)
Rank 0 assigns:    job 5 --> Worker 2

Worker 1 replies:  4     --> Rank 0    (result for job 2)
Rank 0 assigns:    job 6 --> Worker 1

Worker 1 replies:  36    --> Rank 0    (result for job 6)
Worker 2 replies:  25    --> Rank 0    (result for job 5)

Rank 0 returns: [4, 9, 25, 36]         (original job order)
```

Implement `task_queue(jobs, compute)`. Workers call `compute(job)` once for each assigned
integer job, while rank 0 coordinates. Keep assigning work as workers become available.

Return the results in the original job order on rank 0 and `None` on workers.
Each list entry is a separate job, even if values repeat. Use point-to-point communication
through rank 0; you choose how to represent assignments, replies, and shutdown.
Every worker must finish, including those that receive no jobs.

<details>
<summary>Hint: waiting for replies</summary>

`flock.ANY_SOURCE` lets rank 0 accept a reply without choosing a worker in advance.

</details>


In [ ]:
from collections.abc import Callable


async def task_queue(jobs: list[int] | None, compute: Callable[[int], int]) -> list[int] | None:
    if flock.get_rank() == 0:
        assert jobs is not None
        raise NotImplementedError

    assert jobs is None
    raise NotImplementedError


In [ ]:
jobs = [2, 3, 5, 6]
expected = [4, 9, 25, 36]

run = flock.distribute(workers=3, seed=0, trace=True)(task_queue)
results, trace = run(flock.per_rank([jobs, None, None]), compute=lambda job: job * job)
check(results, [expected, None, None])
results

In [ ]:
check_task_queue(task_queue)

## 4. Receive selectively

Imagine several workers running jobs while rank 0 uses their results to carry out its
own computation. Each step needs a particular kind of result, identified by a **tag**.
Workers may produce results in a different order. With limited room to store them,
rank 0 wants to receive just what its current step needs.

Each worker has `messages`: a list of `(tag, value)` pairs to send to rank 0.
Rank 0 has `order`: the tags it needs, in processing order.

```text
Worker 1 has: (tag=1, value=10), (tag=3, value=30)
Worker 2 has: (tag=2, value=20), (tag=4, value=40)

Rank 0 needs tags [2, 1, 3]:

  1. Wait for tag 2   ←   Worker 2 sends 20
  2. Wait for tag 1   ←   Worker 1 sends 10
  3. Wait for tag 3   ←   Worker 1 sends 30

  Return [20, 10, 30]
```

The result tagged `4` is not needed for these steps.

Implement `receive_selectively(order, messages)`. Rank 0 must receive one matching result
for each tag in `order` and return those values in order. Workers return `None`.
There are enough matching results; receiving other results in advance and sorting them
locally doesn't satisfy the task.

Send every supplied result exactly once, unchanged and with its tag. Use point-to-point
communication through rank 0 and leave no unfinished exchanges, including results outside
`order`. Result tags are positive; tag `0` is available for your own control messages.

<details>
<summary>Hint: selecting messages</summary>

Use `tag=...` on sends and receives. `flock.ANY_SOURCE` accepts a matching message from
any worker; `flock.ANY_TAG` accepts any tag.

</details>


In [ ]:
async def receive_selectively(
    order: list[int] | None,
    messages: list[tuple[int, int]] | None,
) -> list[int] | None:
    if flock.get_rank() == 0:
        assert order is not None and messages is None
        raise NotImplementedError

    assert order is None and messages is not None
    raise NotImplementedError


In [ ]:
order = [2, 1, 3]
messages = [[(1, 10), (3, 30)], [(2, 20), (4, 40)]]
run = flock.distribute(workers=3, seed=0, trace=True)(receive_selectively)
results, trace = run(
    flock.per_rank([order, None, None]),
    flock.per_rank([None, *messages]),
)
check(results, [[20, 10, 30], None, None])
results


In [ ]:
check_receive_selectively(receive_selectively)

In [ ]:
from dist_puzzles.lessons.p2p import check_life


## 5. Distributed Game of Life

Imagine a board where each cell is either alive or dead. In Conway's Game of Life,
the board evolves one generation at a time: a cell's next state depends on its eight
neighbours, including diagonals.

- A living cell survives with two or three living neighbours.
- A dead cell becomes alive with exactly three living neighbours.
- Every other cell is dead in the next generation.

All cells update from the same generation. For example, this line alternates between
a vertical and a horizontal position (`#` is alive, `.` is dead):

```text
Generation 0     Generation 1     Generation 2

. . . . .        . . . . .        . . . . .
. . # . .        . . . . .        . . # . .
. . # . .   →    . # # # .   →    . . # . .
. . # . .        . . . . .        . . # . .
. . . . .        . . . . .        . . . . .
. . . . .        . . . . .        . . . . .
```

Now give each rank just one cell. It knows its own state, but needs messages from its
neighbours to work out what happens next. Our board wraps around both edges: stepping
off one side brings you back on the opposite side.

Implement `life(cell, shape, steps)`. Ranks are numbered row by row: rank
`row * width + col` owns cell `(row, col)`. `shape` is `(height, width)`, with both
dimensions at least three. Return your cell's state after `steps` generations.

Each cell is a Python `bool`. Use `flock` for communication. Each generation,
send your current cell once to each of its eight neighbours and receive theirs.
Determine the neighbouring ranks and apply the rules yourself. Finish every exchange;
for zero steps, return the input without communication.

<details>
<summary>Hint: keeping generations separate</summary>

A neighbour may reach the next generation before you do. A generation number can serve
as a message tag.

</details>


In [ ]:
async def life(cell: bool, shape: tuple[int, int], steps: int) -> bool:
    raise NotImplementedError


Run the first update of the board above, passing one boolean cell to each rank.

In [ ]:
board = [
    ".....",
    "..#..",
    "..#..",
    "..#..",
    ".....",
    ".....",
]
expected_board = [
    ".....",
    ".....",
    ".###.",
    ".....",
    ".....",
    ".....",
]

shape = (len(board), len(board[0]))
inputs = [cell == "#" for row in board for cell in row]
expected = [cell == "#" for row in expected_board for cell in row]
run = flock.distribute(workers=len(inputs), seed=0, trace=True)(life)
results, trace = run(flock.per_rank(inputs), shape=shape, steps=1)
check(results, expected)
check_life(life)
for start in range(0, len(results), shape[1]):
    print("".join("#" if cell else "." for cell in results[start:start + shape[1]]))


### Let it run

Edit `initial_board` below (`#` is alive, `.` is dead), then run the animation cell.
Each frame calls your `life` function for one generation. Change `frame_delay` to adjust
the pace. Use the notebook's stop/interrupt button to end the animation; edit the input
and rerun to start again. The board wraps at its edges.


In [ ]:
initial_board = [
    "............",
    "..#.........",
    "...#........",
    ".###........",
    "............",
    "............",
    "............",
    "............",
    "............",
    "............",
    "............",
    "............",
]
frame_delay = 0.2


In [ ]:
import time

from IPython.display import HTML, display

height, width = len(initial_board), len(initial_board[0]) if initial_board else 0
assert height >= 3 and width >= 3, "Use a board with at least three rows and columns."
assert all(len(row) == width for row in initial_board), "All rows must have the same width."
assert all(cell in ".#" for row in initial_board for cell in row), "Use only '.' and '#'."
assert frame_delay > 0, "frame_delay must be positive."

state = [cell == "#" for row in initial_board for cell in row]
advance = flock.distribute(workers=len(state), seed=0)(life)
view = display(HTML(""), display_id=True)
generation = 0

try:
    while True:
        squares = "".join(
            f'<span style="width:18px;height:18px;background:{"#2563eb" if alive else "#e2e8f0"}"></span>'
            for alive in state
        )
        view.update(HTML(
            f'<p>Generation {generation} · {sum(state)} alive</p>'
            f'<div role="img" aria-label="Game of Life generation {generation}" '
            f'style="display:inline-grid;grid-template-columns:repeat({width},18px);gap:1px">'
            f'{squares}</div>'
        ))
        time.sleep(frame_delay)
        state = advance(flock.per_rank(state), shape=(height, width), steps=1)
        generation += 1
except KeyboardInterrupt:
    print("Stopped. Edit initial_board and rerun to start again.")
